# NYC Food / Open-Data Datasets — Explained

This notebook documents every raw dataset stored in `database/datasets/`, the inputs for the
planned NYC food search project.

For each file it covers:

- **What it is** and where it comes from (NYC Open Data)
- **Grain** — what a single row represents — and overall **size**
- **Columns** and what they mean
- **Caveats** — missing values, unit quirks, and type issues to watch for
- A **runnable preview** using pandas

> This is a documentation/exploration notebook. Nothing here writes to the datasets.


## Running this notebook

The preview cells use `pandas`. A project virtual environment is provided:

```bash
python3 -m venv .venv
.venv/bin/python -m pip install pandas jupyter ipykernel
.venv/bin/python -m ipykernel install --user --name datasets-venv --display-name "Python (.venv datasets)"
```

Then select the **Python (.venv datasets)** kernel, or execute headless from the repo root:

```bash
.venv/bin/jupyter nbconvert --to notebook --execute ipynb/datasets_explained.ipynb
```

> **Filename convention:** the `YYYYMMDD` suffix on every file is the **download date**, not the
> data's coverage date. The contents reference older periods (e.g. 2009–2026).


## Datasets at a glance

| File | Format | Rows | Grain | Identifier |
|---|---|---|---|---|
| `restaurant_data_set_2_*.csv` | CSV | 296,623 | one row per inspection violation | `CAMIS` |
| `DOHMH_MenuStat_(Historical)_*.csv` | CSV | 65,219 | one row per menu item per year | `Menu_Item_ID` |
| `Good_Food_Purchasing_Data_*.csv` | CSV | 17,208 | one row per product purchase line | none |
| `NYC_Farmers_Markets_*.csv` | CSV | 2,218 | one row per market per year | `Market Name` + `Year` |
| `Recognized_Shop_Healthy_Stores_*.csv` | CSV | 709 | one row per recognized store | none |
| `DPR_Eateries_001.json` | JSON | 142 | one row per permitted park eatery | `permit_number` |

**No single shared primary key spans these files.** See *Relationships between datasets* near the end.


In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

# Resolve the datasets directory whether the notebook runs from ipynb/ or the repo root.
BASE = Path("../database/datasets")
if not BASE.exists():
    BASE = Path("database/datasets")

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

def profile(df, name):
    """Print a one-line shape summary and preview the first rows."""
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    display(df.head())

print("Reading datasets from:", BASE.resolve())

Reading datasets from: /Users/williamzheng/Documents/2025_stuff/NewYork/NYU/Classes/2026_Semester_3/software_engineering/project/team-3-fall26/database/datasets


---
## 1. Restaurant Inspections — `restaurant_data_set_2_20261011.csv`

Restaurant inspection results and health grades from the NYC Department of Health and
Mental Hygiene (DOHMH). This is the largest file (~74 MB).

- **Grain:** one row per **violation** cited at an inspection. A single inspection of one
  restaurant produces many rows; a clean inspection produces a single row with blank
  violation fields.
- **Identifier:** `CAMIS` — stable restaurant ID (31,516 unique here).
- **Columns:**
  | Column | Meaning |
  |---|---|
  | `CAMIS` | Restaurant identifier |
  | `DBA` | "Doing business as" name |
  | `BORO` | Borough |
  | `STREET` | Street address |
  | `ZIPCODE` | ZIP |
  | `INSPECTION DATE` | `MM/DD/YYYY` (range 01/01/1900–12/31/2025; the 1900 value is a sentinel) |
  | `VIOLATION CODE` | Short code, blank for clean inspections |
  | `VIOLATION DESCRIPTION` | Human-readable violation text |
  | `GRADE` | Inspection grade |
- **Grade values:** `A` (98k rows), `B` (18k), `C` (13k), `N` (not yet graded), `Z`, `P`.
  About **51% of rows have no grade** — a grade is attached only to graded inspection rows,
  not to every violation row.
- **Caveats:** `ZIPCODE` missing in 3,106 rows; `BORO` blank in 346 rows; duplicate
  restaurant/violation rows are common. This is **not** a clean one-row-per-restaurant table.


In [2]:
restaurants = pd.read_csv(BASE / "restaurant_data_set_2_20261011.csv")
profile(restaurants, "restaurant_data_set_2")

restaurant_data_set_2: 296,623 rows x 9 columns


,CAMIS,DBA,BORO,STREET,ZIPCODE,INSPECTION DATE,VIOLATION CODE,VIOLATION DESCRIPTION,GRADE
0,30075445,MORRIS PARK BAKE SHOP,Bronx,MORRIS PARK AVENUE,10462.0,02/27/2026,06B,"Tobacco or electronic cigarette use, eating, o...",A
1,30075445,MORRIS PARK BAKE SHOP,Bronx,MORRIS PARK AVENUE,10462.0,02/27/2026,10F,Non-food contact surface or equipment made of ...,A
2,30075445,MORRIS PARK BAKE SHOP,Bronx,MORRIS PARK AVENUE,10462.0,11/08/2024,06C,"Food, supplies, or equipment not protected fro...",A
3,30075445,MORRIS PARK BAKE SHOP,Bronx,MORRIS PARK AVENUE,10462.0,11/08/2024,08C,Pesticide not properly labeled or used by unli...,A
4,30075445,MORRIS PARK BAKE SHOP,Bronx,MORRIS PARK AVENUE,10462.0,11/08/2024,10F,Non-food contact surface or equipment made of ...,A


In [3]:
print("Unique restaurants (CAMIS):", restaurants["CAMIS"].nunique())
print("Inspection date range:", restaurants["INSPECTION DATE"].min(), "->", restaurants["INSPECTION DATE"].max())
display(restaurants["BORO"].value_counts(dropna=False).to_frame("rows"))
display(restaurants["GRADE"].value_counts(dropna=False).to_frame("rows"))

Unique restaurants (CAMIS): 31516
Inspection date range: 01/01/1900 -> 12/31/2025


,rows
BORO,
Manhattan,109711
Brooklyn,75046
Queens,74491
Bronx,27502
Staten Island,9527
0,346


,rows
GRADE,
NaN,151499
A,98043
B,18015
C,12982
N,10150
Z,5214
P,720


---
## 2. Menu Nutrition Statistics — `DOHMH_MenuStat_(Historical)_20261011.csv`

Menu nutrition statistics for large chain restaurants, keyed by `Restaurant_ID` and
`Menu_Item_ID`. 49 columns, ~21 MB.

- **Grain:** one row per **menu item**, per restaurant, per **year** (`Year` is 2017 or 2018).
- **Traditional key:** `Menu_Item_ID`; the `Restaurant_ID` / `restaurant` pair identifies the chain.
- **Column groups** (beyond identity fields `Menu_Item_ID`, `Year`, `Restaurant_Item_Name`,
  `restaurant`, `Restaurant_ID`, `Item_Name`, `Item_Description`, `Food_Category`):
  - **Serving info:** `Serving_Size`, `Serving_Size_text`, `Serving_Size_Unit`, `Serving_Size_household`
  - **Per-serving nutrients:** `Calories`, `Total_Fat`, `Saturated_Fat`, `Trans_Fat`, `Cholesterol`,
    `Sodium`, `Potassium`, `Carbohydrates`, `Protein`, `Sugar`, `Dietary_Fiber`
  - **Per-100g nutrients:** the same 11 fields suffixed `_100g` (normalized for comparison)
  - **Text mirrors:** the same fields suffixed `_text` (original label strings)
  - **Flags:** `Kids_Meal`, `Limited_Time_Offer`, `Regional`, `Shareable` (0/1)
- **Caveats:** nutrient coverage is incomplete — ~15% of rows miss `Calories` and ~60% miss
  `Calories_100g`. There are 96 distinct `restaurant` names but 94 `Restaurant_ID`s (minor
  naming drift). **`Restaurant_ID` is a separate ID space from the inspection `CAMIS`.**


In [4]:
menustat = pd.read_csv(BASE / "DOHMH_MenuStat_(Historical)_20261011.csv", low_memory=False)
profile(menustat, "DOHMH_MenuStat")

DOHMH_MenuStat: 65,219 rows x 49 columns


,Menu_Item_ID,Year,Restaurant_Item_Name,restaurant,Restaurant_ID,Item_Name,Item_Description,Food_Category,Serving_Size,Serving_Size_text,Serving_Size_Unit,Serving_Size_household,Calories,Total_Fat,Saturated_Fat,Trans_Fat,Cholesterol,Sodium,Potassium,Carbohydrates,Protein,Sugar,Dietary_Fiber,Calories_100g,Total_Fat_100g,Saturated_Fat_100g,Trans_Fat_100g,Cholesterol_100g,Sodium_100g,Potassium_100g,Carbohydrates_100g,Protein_100g,Sugar_100g,Dietary_Fiber_100g,Calories_text,Total_Fat_text,Saturated_Fat_text,Trans_Fat_text,Cholesterol_text,Sodium_text,Potassium_text,Carbohydrates_text,Protein_text,Sugar_text,Dietary_Fiber_text,Kids_Meal,Limited_Time_Offer,Regional,Shareable
0,170040,2018,7 Eleven 100% Colombian Coffee,7 Eleven,1,100% Colombian Coffee,"100% Colombian Coffee, Coffee, Start My Day, T...",Beverages,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,1,0
1,170093,2018,7 Eleven 10 For $2 Mini Tacos,7 Eleven,1,10 For $2 Mini Tacos,"10 For $2 Mini Tacos, Late Night, Deals, Hot S...",Sandwiches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
2,150023,2018,7 Eleven 2 for $1.69 Any Ring Donut,7 Eleven,1,2 for $1.69 Any Ring Donut,"2 for $1.69 Any Ring Donut, Bakery, Start My D...",Baked Goods,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
3,170067,2018,7 Eleven 5 For $4 Grill Mix N Match,7 Eleven,1,5 For $4 Grill Mix N Match,"5 For $4 Grill Mix N Match, Snacks, Hot Snacks...",Entrees,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
4,170112,2018,7 Eleven 6 Mini Breakfast Sausage Empanadas,7 Eleven,1,6 Mini Breakfast Sausage Empanadas,"6 Mini Breakfast Sausage Empanadas w/ Sausage,...",Sandwiches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0


In [5]:
print("Years:", sorted(menustat["Year"].dropna().unique().tolist()))
print("Distinct restaurants:", menustat["restaurant"].nunique(),
      "| distinct Restaurant_ID:", menustat["Restaurant_ID"].nunique())
display(menustat["Food_Category"].value_counts(dropna=False).to_frame("rows").head(12))
missing = menustat[["Calories", "Serving_Size", "Calories_100g"]].isna().mean().to_frame("fraction_missing")
display(missing)

Years: [2017, 2018]
Distinct restaurants: 96 | distinct Restaurant_ID: 94


,rows
Food_Category,
Beverages,23396
Toppings & Ingredients,10153
Entrees,7579
Sandwiches,6391
Appetizers & Sides,4123
Pizza,3373
Desserts,3016
Baked Goods,1908
Salads,1807


,fraction_missing
Calories,0.151858
Serving_Size,0.587559
Calories_100g,0.603214


In [6]:
display(menustat[menustat["Calories"].notna()][
    ["restaurant", "Item_Name", "Food_Category", "Serving_Size", "Calories", "Calories_100g"]
].head())

,restaurant,Item_Name,Food_Category,Serving_Size,Calories,Calories_100g
23,7 Eleven,Blue Raspberry Slurpee,Beverages,12,100,29
46,7 Eleven,Coca-Cola Slurpee,Beverages,12,100,29
47,7 Eleven,Cotton Candy Slurpee,Beverages,12,100,29
103,7 Eleven,Oven Roasted Turkey Sandwich,Sandwiches,NaN,300,NaN
106,7 Eleven,Pina Colada Slurpee,Beverages,12,100,29


---
## 3. Agency Food Purchases — `Good_Food_Purchasing_Data_20261011.csv`

Food purchases made by NYC agencies under the Good Food Purchasing Program (~3.6 MB).

- **Grain:** one row per **product purchase line** (agency × time period × product × vendor).
- **Columns:**
  | Column | Meaning |
  |---|---|
  | `Agency` | Purchasing agency (7 distinct) |
  | `Time Period` | Fiscal period: `2018-2019`, `2019-2020`, `2020-2021` |
  | `Food Product Group` | High-level category (Produce, Meals, Meat, …) |
  | `Food Product Category` | Sub-category |
  | `Product Name` | Generic product |
  | `Product Type` | Specific product / brand variant |
  | `Origin Detail` | Brand or origin |
  | `Distributor` / `Vendor` | Supply chain parties |
  | `# of Units` | Count of units |
  | `Total Weight in lbs` | Weight |
  | `Total Cost` | Cost as a **currency string**, e.g. `"$560"` |
- **Caveats:** `Total Cost` and numeric columns load as text and need cleaning
  (`str.replace` + `pd.to_numeric`). There is no stable product ID.


In [7]:
goodfood = pd.read_csv(BASE / "Good_Food_Purchasing_Data_20261011.csv")
profile(goodfood, "Good_Food_Purchasing_Data")

Good_Food_Purchasing_Data: 17,208 rows x 12 columns


,Agency,Time Period,Food Product Group,Food Product Category,Product Name,Product Type,Origin Detail,Distributor,Vendor,# of Units,Total Weight in lbs,Total Cost
0,Administration for Childrens Services,2018-2019,Meals,Meals,"baby food, dinner, assorted","Stage 2 Baby Food Dinner, Assorted Variety",Beech-Nut,Babylab Inc.,Babylab Inc.,40,100,$560
1,Administration for Childrens Services,2018-2019,Meals,Meals,"baby food, dinner, assorted","Stage 3 Baby Food Dinner, Assorted Variety",Beech-Nut,Babylab Inc.,Babylab Inc.,40,106,$600
2,Administration for Childrens Services,2018-2019,Meals,Meals,"baby food, formula","Premium Nursette Ready-to-Feed Infant Formula,...",Enfamil,Babylab Inc.,Babylab Inc.,200,"1,200","$16,200"
3,Administration for Childrens Services,2018-2019,Beverages,Beverages,"juice, fruit punch, ss","100% Juice Variety Pack, 4 fl oz bottles",Enfamil,Babylab Inc.,Babylab Inc.,100,600,"$1,940"
4,Administration for Childrens Services,2018-2019,Condiments & Snacks,Condiments & Snacks,"cereal, baby, oatmeal",Single Grain Oatmeal Baby Cereal,Enfamil,Babylab Inc.,Babylab Inc.,10,60,$290


In [8]:
display(goodfood["Food Product Group"].value_counts(dropna=False).to_frame("rows"))

# Total Cost arrives as "$1,234" strings -> parse to a numeric column.
goodfood["cost_usd"] = (
    goodfood["Total Cost"].astype(str)
    .str.replace(r"[$,]", "", regex=True)
    .replace({"nan": None})
    .astype(float)
)
display(goodfood[["Agency", "Time Period", "Total Cost", "cost_usd"]].head())
display(goodfood.groupby("Agency")["cost_usd"].sum().sort_values(ascending=False).to_frame("total_cost_usd"))

,rows
Food Product Group,
Condiments & Snacks,4881
Produce,3821
Meals,2086
Meat,1786
"Bread, Grains & Legumes",1557
Beverages,1396
Milk & Dairy,1297
Seafood,383
Non-Food,1


,Agency,Time Period,Total Cost,cost_usd
0,Administration for Childrens Services,2018-2019,$560,560.0
1,Administration for Childrens Services,2018-2019,$600,600.0
2,Administration for Childrens Services,2018-2019,"$16,200",16200.0
3,Administration for Childrens Services,2018-2019,"$1,940",1940.0
4,Administration for Childrens Services,2018-2019,$290,290.0


,total_cost_usd
Agency,
Department of Education,494343992.0
Department of Correction,17506267.0
Human Resources Administration,13401745.0
Administration for Children's Services,1414939.0
Administration for Childrens Services,162858.0
Department of Homeless Services,0.0
Health + Hospitals,0.0


---
## 4. Farmers Markets — `NYC_Farmers_Markets_20261011.csv`

Locations and operating schedules of NYC farmers markets (~300 KB).

- **Grain:** one row per **market per year** (`Year` spans 2009–2026).
- **Columns:** `Year`, `Market Name`, `Borough`, `Street Address`, `Community District`,
  `Latitude`, `Longitude`, `Days of Operation`, `Hours of Operations`, `Season Dates`,
  `Accepts EBT`, `Open Year-Round`, `Stellar Cooking Demonstrations`.
- **Useful for mapping:** `Latitude`/`Longitude` support geospatial joins.
- **Caveats:** ~852 rows have no coordinates. `Accepts EBT` is inconsistent
  (`Yes` / `No` / blank / lowercase `yes`) and needs normalization. `Community District`
  is a coded value (e.g. `112`, `307`).


In [9]:
markets = pd.read_csv(BASE / "NYC_Farmers_Markets_20261011.csv")
profile(markets, "NYC_Farmers_Markets")

NYC_Farmers_Markets: 2,218 rows x 13 columns


,Year,Market Name,Borough,Street Address,Community District,Latitude,Longitude,Days of Operation,Hours of Operations,Season Dates,Accepts EBT,Open Year-Round,Stellar Cooking Demonstrations
0,2025,175th Street Greenmarket,Manhattan,W. 175th St. bet. Wadsworth Ave. & Broadway,112.0,40.845948,-73.937811,Thursday,8 a.m. - 4p.m.,NaN,Yes,No,No
1,2025,4th Ave Sunset Park Greenmarket,Brooklyn,4th Ave. 60th St.,307.0,40.647975,-74.003744,Saturday,8 a.m. - 3 p.m.,NaN,Yes,No,No
2,2025,57th Street Greenmarket,Manhattan,57th St. & 10th Ave.,104.0,40.769117,-73.988559,Saturday,8 a.m. - 3 p.m.,NaN,Yes,No,No
3,2025,79th Street Greenmarket,Manhattan,79th St. & Columbus Ave.,107.0,40.782295,-73.975244,Sunday,9 a.m. - 4 p.m,NaN,Yes,Yes,No
4,2025,7th Ave Sunset Park Greenmarket,Brooklyn,7th Ave. & 44th St.,307.0,40.645841,-74.002402,Saturday,8 a.m. - 3 p.m.,NaN,Yes,Yes,No


In [10]:
print("Years covered:", sorted(markets["Year"].dropna().unique().tolist()))
display(markets["Borough"].value_counts(dropna=False).to_frame("rows"))
display(markets["Accepts EBT"].value_counts(dropna=False).to_frame("rows"))
print("Rows missing coordinates:", int(markets[["Latitude", "Longitude"]].isna().any(axis=1).sum()))

Years covered: [2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


,rows
Borough,
Brooklyn,712
Manhattan,690
Bronx,445
Queens,313
Staten Island,58


,rows
Accepts EBT,
Yes,1307
NaN,853
No,56
yes,2


Rows missing coordinates: 856


In [11]:
display(markets[["Market Name", "Borough", "Days of Operation", "Accepts EBT", "Latitude", "Longitude"]].head(8))

,Market Name,Borough,Days of Operation,Accepts EBT,Latitude,Longitude
0,175th Street Greenmarket,Manhattan,Thursday,Yes,40.845948,-73.937811
1,4th Ave Sunset Park Greenmarket,Brooklyn,Saturday,Yes,40.647975,-74.003744
2,57th Street Greenmarket,Manhattan,Saturday,Yes,40.769117,-73.988559
3,79th Street Greenmarket,Manhattan,Sunday,Yes,40.782295,-73.975244
4,7th Ave Sunset Park Greenmarket,Brooklyn,Saturday,Yes,40.645841,-74.002402
5,82nd Street Greenmarket,Manhattan,Saturday,Yes,40.774292,-73.951262
6,97th Street Greenmarket,Manhattan,Friday,Yes,40.794003,-73.968070
7,Abingdon Square Park Greenmarket,Manhattan,Saturday,Yes,40.737553,-74.005687


---
## 5. Recognized Healthy Stores — `Recognized_Shop_Healthy_Stores_20261011.csv`

Stores recognized under the Shop Healthy NYC program (~100 KB).

- **Grain:** one row per recognized **store**.
- **Columns:** `Store Name`, `Street Address`, `Borough`, `Zip Code`, `Year Awarded`,
  `Program Wave`, `Latitude`, `Longitude`, `Community Board`, `Council District`, `BIN`,
  `BBL`, `Census Tract (2020)`, `Neighborhood Tabulation Area (NTA) (2020)`.
- **Note:** the raw CSV header contains **embedded newlines** (`"Zip 
Code"`,
  `"Program 
Wave"`). The cell below strips them so column access is clean.
- **Caveats:** `Borough` mixes `"New York"` and `"Manhattan"` for the same borough;
  `BIN`/`BBL`/census values load as strings with thousands separators and need cleaning.
  `Year Awarded` spans 2013–2024.


In [12]:
stores = pd.read_csv(BASE / "Recognized_Shop_Healthy_Stores_20261011.csv")
# Collapse the newlines that are embedded in the raw header.
stores.columns = [c.replace("\n", " ").strip() for c in stores.columns]
profile(stores, "Recognized_Shop_Healthy_Stores")

Recognized_Shop_Healthy_Stores: 709 rows x 14 columns


,Store Name,Street Address,Borough,Zip Code,Year Awarded,Program Wave,Latitude,Longitude,Community Board,Council District,BIN,BBL,Census Tract (2020),Neighborhood Tabulation Area (NTA) (2020)
0,Rivas Deli Grocery,651 East 183rd Street,Bronx,10458,2013,1,40.852568,-73.887484,206.0,15.0,"2,012,464","2,030,870,033",391.0,BX0603
1,C-Town,1761 Southern Boulevard,Bronx,10460,2013,1,40.837267,-73.888016,203.0,15.0,"2,009,878","2,029,400,047",155.0,BX0303
2,La India Mini Market,1942 Marmion Avenue,Bronx,10460,2013,1,40.842942,-73.887136,206.0,15.0,"2,010,156","2,029,600,059",36502.0,BX0602
3,Los Hermanos,2019 Vyse Avenue,Bronx,10460,2013,1,40.842391,-73.881835,206.0,15.0,"2,013,203","2,031,270,020",363.0,BX0601
4,Price Choice Food Market,820 East 180th Street,Bronx,10460,2013,1,40.846173,-73.884871,206.0,15.0,"2,012,955","2,031,080,008",36501.0,BX0602


In [13]:
print("Columns:", list(stores.columns))
display(stores["Borough"].value_counts(dropna=False).to_frame("rows"))
print("Year Awarded range:", stores["Year Awarded"].min(), "->", stores["Year Awarded"].max())

Columns: ['Store Name', 'Street Address', 'Borough', 'Zip  Code', 'Year Awarded', 'Program  Wave', 'Latitude', 'Longitude', 'Community Board', 'Council District', 'BIN', 'BBL', 'Census Tract (2020)', 'Neighborhood Tabulation Area (NTA) (2020)']


,rows
Borough,
Bronx,312
Brooklyn,245
New York,139
Manhattan,13


Year Awarded range: 2013 -> 2024


---
## 6. Park Eateries (JSON) — `DPR_Eateries_001.json`

Concessions and food vendors permitted in NYC Department of Parks & Recreation parks
(~53 KB). Unlike the CSVs, this is a **JSON array**.

- **Grain:** one object per permitted **eatery / vendor** (142 objects).
- **Fields:** `name`, `location`, `park_id`, `start_date`, `end_date`, `description`,
  `permit_number`, `phone`, `website`, `type_name`.
- **`type_name` values:** Mobile Food Truck (51), Food Cart (42), Snack Bar (25),
  Restaurant (12), Specialty Cart (11), Breakfast Cart (1).
- **Caveats:** `start_date`, `end_date`, `description`, and `phone` are frequently `null`;
  location is free text (no coordinates), but `park_id` (e.g. `M010`) encodes the park.


In [14]:
with open(BASE / "DPR_Eateries_001.json") as f:
    eateries = pd.DataFrame(json.load(f))

profile(eateries, "DPR_Eateries")

DPR_Eateries: 142 rows x 10 columns


,name,location,park_id,start_date,end_date,description,permit_number,phone,website,type_name
0,King David Tacos,Prospect Park,B073,None,2026-11-09,None,B73-O,None,https://www.prospectpark.org/visit-the-park/pa...,Breakfast Cart
1,Central Park Food Cart,"Central Park Mall Area, northwest side of the ...",M010,2023-07-26,2027-12-31,None,M10-72-1A-C,None,None,Food Cart
2,NUTS ABOVE NUTS INC,East Drive and 72 Street,M010,2025-10-01,2029-12-31,None,M10-72-ED-C,None,None,Food Cart
3,"SHAWON TRADE, INC.","West Drive, between West 74 and West 75 streets",M010,2025-03-11,2029-12-31,None,M10-74-WD-C,None,None,Food Cart
4,Flushing Meadows Corona Park Food Cart,North corner of Soccer Field #5,Q099,2023-09-28,2027-12-31,None,Q99-(1)-C,None,None,Food Cart


In [15]:
display(eateries["type_name"].value_counts(dropna=False).to_frame("rows"))
print("Distinct parks (park_id):", eateries["park_id"].nunique())
display(eateries[["name", "location", "park_id", "permit_number", "type_name"]].head())

,rows
type_name,
Mobile Food Truck,51
Food Cart,42
Snack Bar,25
Restaurant,12
Specialty Cart,11
Breakfast Cart,1


Distinct parks (park_id): 72


,name,location,park_id,permit_number,type_name
0,King David Tacos,Prospect Park,B073,B73-O,Breakfast Cart
1,Central Park Food Cart,"Central Park Mall Area, northwest side of the ...",M010,M10-72-1A-C,Food Cart
2,NUTS ABOVE NUTS INC,East Drive and 72 Street,M010,M10-72-ED-C,Food Cart
3,"SHAWON TRADE, INC.","West Drive, between West 74 and West 75 streets",M010,M10-74-WD-C,Food Cart
4,Flushing Meadows Corona Park Food Cart,North corner of Soccer Field #5,Q099,Q99-(1)-C,Food Cart


---
## Relationships between datasets

There is **no single shared primary key** across these files. Joins must be built
deliberately:

- **`CAMIS`** (inspections) and **`Restaurant_ID`** (MenuStat) are **distinct ID spaces**.
  Joining them requires fuzzy matching on `DBA` / `restaurant` names — no exact key.
- **Geospatial joins** are the most reliable common ground. Farmers markets and recognized
  stores both carry `Latitude`/`Longitude`, so they can be joined to each other (and to any
  geocoded restaurant address) by distance.
- **`park_id`** in the eateries JSON can link eateries within the same park.
- **Borough** is present in most datasets, but values differ in spelling
  (`"New York"` vs `"Manhattan"`) and NYPD-style codes, so normalize before grouping.
- **No key** links Good Food purchasing rows to any other dataset.

| Join | Key / method | Reliability |
|---|---|---|
| Inspections ↔ MenuStat | name fuzzy match | low |
| Markets ↔ Stores | lat/long distance | high |
| Eateries ↔ Parks | `park_id` | high |
| Anything ↔ geography | geocoding | medium |


---
## How these map to the project (see `TODO.md`)

| Goal | Relevant datasets |
|---|---|
| Consolidate NYC Open Data sources | All six files, unified behind a common schema |
| Search-results website | Inspections (grades) + MenuStat (nutrition) for search facets |
| Vector database | Free-text fields (`Item_Description`, `VIOLATION DESCRIPTION`, `Product Name`) for embeddings |
| Django pipeline | Loaders per file → models; lat/long + `park_id` for map features |

Key takeaways before building:

1. Data is **messy** — plan for cleaning (currency strings, newline headers, inconsistent
   booleans) in the ingestion layer.
2. **Grain differs per file** — normalize to consistent entities (restaurant, market, store,
   eatery) before searching.
3. **IDs don't line up** — design a canonical entity/key strategy (e.g. your own surrogate
   IDs plus geospatial linkage) rather than relying on source keys.
